# 12 — Annotation en aveugle : l'écart de registre ne survit pas à l'étiquetage

Le [notebook 11](11_corpus_etendu.ipynb) a laissé une question ouverte, et une seule.
L'appartenance à une catégorie de Wikipédia s'était révélée un **indicateur bruité** du
registre émotionnel — « Lil Tay » figure dans une catégorie de canulars, mais son audience est
celle d'une célébrité. Or un bruit d'étiquetage attire tout écart vers zéro. Le résultat nul
du corpus étendu était donc compatible avec deux lectures que ces données ne séparaient pas :
absence d'effet, ou effet dilué.

Ce notebook les sépare. Les 440 sujets ont été **annotés à la main**, à partir du seul couple
titre + chapeau d'article, sous une grille [pré-enregistrée](../docs/annotation.md) et
antérieure à toute annotation.

**Ce que l'annotation établit :**

* le bruit d'étiquetage est **massif** — 40 % des sujets ne relèvent d'aucun des deux
  registres, et l'accord entre catégorie et annotation n'est que de 59,5 % ;
* l'écart de taux de basculement **disparaît complètement** : 8,6 % contre 2,7 % devient
  **4,8 % contre 5,1 %**, rapport de cotes 0,93, $p = 1{,}00$ ;
* l'écart de persistance reste nul, ×3,04 contre ×2,90, $p = 0{,}90$, et le test conserve la
  puissance de détecter un écart de l'ampleur annoncée par le corpus pilote ;
* et l'annotation expose un **défaut de conception** que ni le pilote ni le corpus étendu
  n'avaient pu voir : correctement étiquetés, les deux registres ne portent presque pas sur
  les mêmes types de sujets.

## 1. Ce que l'annotation ajoute, et ce qu'elle ne change pas

L'annotation **ne modifie pas le corpus** : elle lui ajoute une colonne. Le pool reste celui
que les dix-sept catégories déclarées ont produit, et les 440 sujets sont annotés sans
exception — annoter un sous-ensemble choisi rouvrirait le biais de sélection que le corpus
dérivé de catégories avait fermé.

La grille pose une seule question : **qu'est-ce qui mobiliserait l'attention du public sur cet
article ?** Non pas de quoi l'article parle, mais quelle émotion porterait sa consultation.

| Registre | Définition |
|---|---|
| `accusation` | une faute, une menace ou une tromperie attribuée à quelqu'un |
| `discovery` | une découverte, une exploration ou une réussite |
| `neither` | ni l'un ni l'autre — divertissement, célébrité, institution ordinaire, entrée de catalogue |

C'est la troisième étiquette qui fait le travail, et le taux auquel elle est employée **mesure**
le bruit d'étiquetage que le notebook 11 n'avait pu que diagnostiquer.

In [1]:
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from ide.annotation import (
    ANNOTATIONS_PATH,
    CONTAMINATED,
    KINDS,
    REGISTERS,
    RUBRIC_VERSION,
    confusion_matrix,
    load_annotations,
)
from ide.catalogue import load_catalogue
from ide.pageviews import load_cached
from ide.plotting import PALETTE, save_figure, use_project_style
from ide.regime import scan_regime_shifts

use_project_style()

entries, report = load_catalogue()
annotations = load_annotations()

print(f"grille           : version {RUBRIC_VERSION}")
print(f"manifeste        : {len(entries)} sujets")
print(f"annotations      : {len(annotations)}  ({ANNOTATIONS_PATH.name})")

counts = Counter(item.register for item in annotations.values())
print("\nregistre annoté")
for register in REGISTERS:
    print(f"  {register:11s} {counts[register]:3d}  ({100 * counts[register] / len(annotations):4.1f} %)")

confidences = Counter(item.confidence for item in annotations.values())
print(f"\nannotations incertaines : {confidences['unsure']} sur {len(annotations)}"
      f"  (toutes justifiées par écrit)")

grille           : version 1.0
manifeste        : 440 sujets
annotations      : 440  (annotations.json)

registre annoté
  accusation  147  (33.4 %)
  discovery   118  (26.8 %)
  neither     175  (39.8 %)

annotations incertaines : 79 sur 440  (toutes justifiées par écrit)


## 2. Le bruit d'étiquetage, mesuré

Première mesure, et elle est un résultat en soi : **à quel point l'appartenance à une catégorie
prédit-elle le registre ?**

In [2]:
matrix = confusion_matrix(entries, annotations)

other = "ni l'un ni l'autre"
print(f"{'catégorie':12s} {'n':>5s} {'accusation':>14s} {'découverte':>14s} {other:>20s}")
print("-" * 70)
for category in ("accusation", "discovery"):
    row = matrix[category]
    total = sum(row.values())
    cells = [f"{row[r]:4d} ({100 * row[r] / total:4.1f} %)" for r in REGISTERS]
    print(f"{category:12s} {total:5d} {cells[0]:>14s} {cells[1]:>14s} {cells[2]:>20s}")

agreement = sum(1 for e in entries if annotations[e.label].register == e.category)
crossed = sum(1 for e in entries
              if annotations[e.label].register not in (e.category, "neither"))

print(f"\naccord catégorie / annotation : {agreement}/{len(entries)}"
      f" = {100 * agreement / len(entries):.1f} %")
print(f"registre franchement inversé  : {crossed}")

catégorie        n     accusation     découverte   ni l'un ni l'autre
----------------------------------------------------------------------
accusation     220   147 (66.8 %)     3 ( 1.4 %)          70 (31.8 %)
discovery      220     0 ( 0.0 %)   115 (52.3 %)         105 (47.7 %)

accord catégorie / annotation : 262/440 = 59.5 %
registre franchement inversé  : 3


### Lecture

**Deux sujets sur cinq ne relèvent d'aucun des deux registres.** Le bruit n'est pas marginal,
il est majoritaire dans un cas sur trois côté accusation et un cas sur deux côté découverte.

Il est aussi **asymétrique**, et pour une raison de construction : le registre « découverte »
tirait ses effectifs de catalogues d'objets célestes, dont l'immense majorité sont des entrées
techniques sans public. Le filtre de substance de dix mille octets n'y suffisait pas.

En revanche, le registre franchement inversé est **quasi nul** : la catégorie se trompe en
capturant des sujets hors registre, presque jamais en attribuant le mauvais registre. C'est
exactement le profil d'un bruit qui dilue sans biaiser.

In [3]:
profiles, shifts = [], []
for entry in entries:
    series = load_cached(entry.project, entry.article)
    if series is None:
        continue
    values = np.clip(series.filled(), 1.0, None)
    try:
        outcome = scan_regime_shifts(values, label=entry.label)
    except ValueError:
        continue

    annotation = annotations[entry.label]
    record = {
        "category": entry.category,
        "register": annotation.register,
        "kind": annotation.kind,
        "confidence": annotation.confidence,
        "subject": entry.label,
        "traffic": float(np.median(values)),
        "shifted": bool(outcome.shifts),
    }
    profiles.append(record)
    for shift in outcome.shifts:
        shifts.append({**record,
                       "date": series.day(shift.index),
                       "lift": shift.lift,
                       "before": shift.level_before,
                       "after": shift.level_after,
                       "identified": shift.has_identified_parameters})

print(f"sujets analysés      : {len(profiles)}")
print(f"changements détectés : {len(shifts)}")
print(f"dont identifiés      : {sum(s['identified'] for s in shifts)}")

sujets analysés      : 440
changements détectés : 28
dont identifiés      : 1


## 3. Le taux de basculement : l'écart s'évanouit

Le corpus étendu avait trouvé ici son seul écart significatif — avant de montrer qu'il suivait
le trafic. L'annotation permet de trancher autrement : en corrigeant l'étiquette elle-même.

In [4]:
def switching_rate(group):
    hits = sum(item["shifted"] for item in group)
    return hits, len(group), (100.0 * hits / len(group) if group else 0.0)


def compare(group_a, group_d):
    ha, na, pa = switching_rate(group_a)
    hd, nd, pd = switching_rate(group_d)
    test = stats.fisher_exact([[ha, na - ha], [hd, nd - hd]])
    return (f"accusation {ha:3d}/{na:3d} = {pa:4.1f} %   découverte {hd:3d}/{nd:3d} = {pd:4.1f} %"
            f"   RC = {test.statistic:5.2f}   p = {test.pvalue:.3f}"), test


for key, label in (("category", "étiquette de catégorie"), ("register", "annotation manuelle")):
    line, _ = compare([p for p in profiles if p[key] == "accusation"],
                      [p for p in profiles if p[key] == "discovery"])
    print(f"{label:24s} {line}")

hits, total, percent = switching_rate([p for p in profiles if p["register"] == "neither"])
print(f"{"(ni l'un ni l'autre)":24s} {hits:3d}/{total:3d} = {percent:4.1f} %")

étiquette de catégorie   accusation  19/220 =  8.6 %   découverte   6/220 =  2.7 %   RC =  3.37   p = 0.012
annotation manuelle      accusation   7/147 =  4.8 %   découverte   6/118 =  5.1 %   RC =  0.93   p = 1.000
(ni l'un ni l'autre)      12/175 =  6.9 %


### Lecture

**Le rapport de cotes passe de 3,4 à 0,93.** L'écart ne s'atténue pas, il disparaît — et
change même très légèrement de sens.

Les sujets écartés comme « ni l'un ni l'autre » basculent à 6,9 %, c'est-à-dire **plus souvent
que les deux registres**. Ils n'étaient pas du bruit inerte : ce sont eux qui portaient l'écart
attribué au registre d'accusation.

In [5]:
accusation = [p for p in profiles if p["register"] == "accusation"]
discovery = [p for p in profiles if p["register"] == "discovery"]

traffic = {
    "catégorie": ([p["traffic"] for p in profiles if p["category"] == "accusation"],
                  [p["traffic"] for p in profiles if p["category"] == "discovery"]),
    "annotation": ([p["traffic"] for p in accusation], [p["traffic"] for p in discovery]),
}
for label, (values_a, values_d) in traffic.items():
    test = stats.mannwhitneyu(values_a, values_d, alternative="two-sided")
    print(f"{label:11s} trafic médian : accusation {np.median(values_a):6.1f}   "
          f"découverte {np.median(values_d):6.1f}   p = {test.pvalue:.1e}")

print("\nLe déséquilibre d'audience était lui-même un effet de l'étiquetage :")
print("les entrées de catalogue sans public gonflaient le registre « découverte ».")

THRESHOLD = 47.0
line, _ = compare([p for p in accusation if p["traffic"] >= THRESHOLD],
                  [p for p in discovery if p["traffic"] >= THRESHOLD])
print(f"\nstrate ≥ {THRESHOLD:.0f} vues/jour   {line}")

pool = sorted(discovery, key=lambda item: item["traffic"])
pool_logs = np.log([item["traffic"] for item in pool])
used, pairs = set(), []
for probe in sorted(accusation, key=lambda item: -item["traffic"]):
    candidates = [(abs(np.log(probe["traffic"]) - pool_logs[i]), i)
                  for i in range(len(pool)) if i not in used]
    if not candidates:
        break
    distance, index = min(candidates)
    if distance <= np.log(2.0):
        used.add(index)
        pairs.append((probe, pool[index]))

only_a = sum(1 for x, y in pairs if x["shifted"] and not y["shifted"])
only_d = sum(1 for x, y in pairs if y["shifted"] and not x["shifted"])
mcnemar = stats.binomtest(only_a, only_a + only_d, 0.5) if only_a + only_d else None
print(f"appariement sur le trafic : {len(pairs)} paires, discordantes {only_a}/{only_d},"
      f" McNemar p = {mcnemar.pvalue:.3f}")

catégorie   trafic médian : accusation   39.0   découverte   11.0   p = 4.8e-14
annotation  trafic médian : accusation   36.0   découverte   26.5   p = 2.6e-03

Le déséquilibre d'audience était lui-même un effet de l'étiquetage :
les entrées de catalogue sans public gonflaient le registre « découverte ».

strate ≥ 47 vues/jour   accusation   5/ 68 =  7.4 %   découverte   6/ 40 = 15.0 %   RC =  0.45   p = 0.323
appariement sur le trafic : 116 paires, discordantes 5/4, McNemar p = 1.000


## 4. La persistance : toujours nulle, et cette fois avec la puissance de le dire

C'est le test principal, celui que le corpus pilote avait cru remporter.

In [6]:
def persistence(source, label):
    lifts_a = np.array([s["lift"] for s in source if s["register"] == "accusation"])
    lifts_d = np.array([s["lift"] for s in source if s["register"] == "discovery"])
    test = stats.mannwhitneyu(lifts_a, lifts_d, alternative="two-sided")
    left = f"×{np.median(lifts_a):.2f} (n={lifts_a.size:2d})"
    right = f"×{np.median(lifts_d):.2f} (n={lifts_d.size:2d})"
    print(f"  {label:34s} {left:>14s} {right:>14s} {test.pvalue:8.3f}")


print(f"{'corpus':36s} {'accusation':>14s} {'découverte':>14s} {'p':>8s}")
print("-" * 74)
print(f"  {'pilote, 24 sujets choisis à la main':34s} {'×9.20 (n= 8)':>14s}"
      f" {'×2.90 (n= 6)':>14s} {0.081:8.3f}")
print(f"  {'étendu, étiquette de catégorie':34s} {'×3.04 (n=21)':>14s}"
      f" {'×2.90 (n= 7)':>14s} {0.533:8.3f}")
persistence(shifts, "étendu, registre annoté")

print("\ncontrôles de sensibilité")
persistence([s for s in shifts if s["subject"] not in CONTAMINATED],
            "sans les sujets contaminés")
persistence([s for s in shifts if s["confidence"] == "sure"],
            "sans les annotations incertaines")
persistence([s for s in shifts if s["traffic"] >= THRESHOLD],
            "strate de trafic comparable")

neither = np.array([s["lift"] for s in shifts if s["register"] == "neither"])
label_other = f"×{np.median(neither):.2f} (n={neither.size:2d})"
print(f"\n  {"(ni l'un ni l'autre)":34s} {label_other:>14s}")

corpus                                   accusation     découverte        p
--------------------------------------------------------------------------
  pilote, 24 sujets choisis à la main   ×9.20 (n= 8)   ×2.90 (n= 6)    0.081
  étendu, étiquette de catégorie       ×3.04 (n=21)   ×2.90 (n= 7)    0.533
  étendu, registre annoté              ×3.04 (n= 7)   ×2.90 (n= 7)    0.902

contrôles de sensibilité
  sans les sujets contaminés           ×2.87 (n= 6)   ×2.90 (n= 7)    0.945
  sans les annotations incertaines     ×3.06 (n= 6)   ×2.90 (n= 5)    0.931
  strate de trafic comparable          ×2.69 (n= 5)   ×2.90 (n= 7)    0.876

  (ni l'un ni l'autre)                 ×3.20 (n=14)


### Un résultat nul n'a de valeur que si le test pouvait détecter quelque chose

Quatorze observations, sept de chaque côté : il faut le dire avant qu'on ne l'objecte. Mais un
test de Mann-Whitney à sept contre sept n'est pas aveugle — il détecte une séparation nette, et
c'est précisément ce que le corpus pilote annonçait, avec des intervalles interquartiles
[4,0 ; 14,9] contre [2,7 ; 3,2] presque disjoints.

In [7]:
from math import comb

print(f"p minimal atteignable à n = 7 contre 7 : {2 / comb(14, 7):.1e}  (séparation complète)\n")
print(f"{'rangs de chevauchement':26s} {'p':>8s}")
print("-" * 36)
for overlap in range(6):
    lower = np.arange(7, dtype=float)
    upper = np.arange(7, dtype=float) + 7 - overlap
    pvalue = stats.mannwhitneyu(upper, lower, alternative="two-sided").pvalue
    verdict = "détecté" if pvalue < 0.05 else ""
    print(f"{overlap:^26d} {pvalue:8.4f}  {verdict}")

print("\nUn écart de l'ampleur annoncée par le corpus pilote aurait été détecté ici.")
print("Le résultat nul n'est donc pas un simple manque de puissance.")

p minimal atteignable à n = 7 contre 7 : 5.8e-04  (séparation complète)

rangs de chevauchement            p
------------------------------------
            0                0.0006  détecté
            1                0.0026  détecté
            2                0.0048  détecté
            3                0.0124  détecté
            4                0.0400  détecté
            5                0.1395  

Un écart de l'ampleur annoncée par le corpus pilote aurait été détecté ici.
Le résultat nul n'est donc pas un simple manque de puissance.


## 5. Ce que l'annotation révèle du plan d'expérience

C'est la découverte que ni le pilote ni le corpus étendu ne pouvaient faire, faute d'étiquette
fiable : **une fois correctement étiquetés, les deux registres ne portent presque pas sur les
mêmes types de sujets.**

In [8]:
kinds_by_register = {
    register: Counter(a.kind for a in annotations.values() if a.register == register)
    for register in REGISTERS
}

print(f"{'type':14s}" + "".join(f"{r:>13s}" for r in REGISTERS))
print("-" * 55)
for kind in KINDS:
    print(f"{kind:14s}" + "".join(f"{kinds_by_register[r][kind]:13d}" for r in REGISTERS))

print("\ntaux de basculement à type de sujet constant")
for kind in KINDS:
    group_a = [p for p in accusation if p["kind"] == kind]
    group_d = [p for p in discovery if p["kind"] == kind]
    if len(group_a) >= 10 and len(group_d) >= 10:
        line, _ = compare(group_a, group_d)
        print(f"  {kind:14s} {line}")
    else:
        print(f"  {kind:14s} accusation n={len(group_a):3d}  découverte n={len(group_d):3d}"
              f"   — effectifs insuffisants pour comparer")

type             accusation    discovery      neither
-------------------------------------------------------
event                    58            6            1
person                   13           39           12
organisation             11            0           13
work                      2            0           19
concept                  63           15           21
object                    0           58          109

taux de basculement à type de sujet constant
  event          accusation n= 58  découverte n=  6   — effectifs insuffisants pour comparer
  person         accusation   3/ 13 = 23.1 %   découverte   3/ 39 =  7.7 %   RC =  3.60   p = 0.157
  organisation   accusation n= 11  découverte n=  0   — effectifs insuffisants pour comparer
  work           accusation n=  2  découverte n=  0   — effectifs insuffisants pour comparer
  concept        accusation   0/ 63 =  0.0 %   découverte   0/ 15 =  0.0 %   RC =   nan   p = 1.000
  object         accusation n=  0  découv

### Lecture

Le registre d'accusation est fait de **concepts et d'événements** ; celui de découverte,
d'**objets et de personnes**. Un seul type de sujet — les personnes — se trouve des deux côtés
en nombre suffisant pour une comparaison, et il n'y donne rien de concluant.

La conséquence est structurelle et dépasse ce corpus : **une comparaison entre registres bâtie
sur des catégories thématiques compare aussi, et peut-être surtout, des natures d'objets.** Un
concept encyclopédique — « Corruption au Mexique » — n'a pas la dynamique d'attention d'un
événement daté, indépendamment de toute charge émotionnelle. Le notebook 11 avait nommé ce
confondant sans pouvoir le mesurer ; il est ici mesuré, et il est sévère.

Un troisième protocole devra donc apparier sur le **type de sujet** autant que sur le trafic.

## 6. Les basculements retenus

Vingt-huit changements de régime, et la moitié d'entre eux portent sur des sujets qu'aucun des
deux registres ne revendique.

In [9]:
print(f"{'annoté':12s} {'catég.':7s} {'sujet':40s} {'date':12s} {'avant→après':>18s} {'':>7s}")
print("-" * 100)
for shift in sorted(shifts, key=lambda item: -item["lift"]):
    flag = "  ← identifié" if shift["identified"] else ""
    print(f"{shift['register']:12s} {shift['category'][:6]:7s} {shift['subject'][:40]:40s} "
          f"{str(shift['date']):12s} {shift['before']:7.0f} → {shift['after']:7.0f}"
          f"   ×{shift['lift']:5.1f}{flag}")

top = sorted(shifts, key=lambda item: -item["lift"])[:5]
print(f"\nSur les cinq plus fortes élévations, {sum(1 for s in top if s['register'] == 'neither')}"
      f" portent sur des sujets codés « ni l'un ni l'autre ».")
print("Ce sont exactement les cas que le notebook 11 avait cités comme suspects.")

annoté       catég.  sujet                                    date                avant→après        
----------------------------------------------------------------------------------------------------
neither      accusa  Lil Tay                                  2020-06-27       290 →    2290   ×  7.9
neither      accusa  Watch Dogs (video game)                  2020-12-06        57 →     358   ×  6.3  ← identifié
neither      accusa  Million Dollar Extreme                   2016-07-06       111 →     631   ×  5.7
neither      accusa  The Capture (TV series)                  2022-07-30       388 →    2133   ×  5.5
discovery    discov  David Baker (biochemist)                 2024-09-24        61 →     330   ×  5.4
accusation   accusa  Mossack Fonseca                          2019-10-14       286 →    1378   ×  4.8
neither      accusa  Illuminati (game)                        2020-01-16       208 →     737   ×  3.5
neither      accusa  Herbert Kickl                            2024-08-

In [10]:
figure, axes = plt.subplots(2, 2, figsize=(11.5, 7.8))
noise, rates, persist, design = axes.ravel()

# (a) Le bruit d'étiquetage : décomposition de chaque catégorie.
categories = ["accusation", "discovery"]
colours = {"accusation": PALETTE["field"], "discovery": PALETTE["remedy"],
           "neither": PALETTE["neutral"]}
names = {"accusation": "accusation", "discovery": "découverte", "neither": "ni l'un ni l'autre"}
bottom = np.zeros(2)
for register in REGISTERS:
    heights = np.array([matrix[c][register] for c in categories], dtype=float)
    noise.bar(np.arange(2), heights, 0.55, bottom=bottom, color=colours[register],
              alpha=0.85, label=f"annoté {names[register]}")
    for index, (height, base) in enumerate(zip(heights, bottom, strict=True)):
        if height >= 20:
            noise.text(index, base + height / 2, f"{height:.0f}", ha="center", va="center",
                       fontsize=8.5, color="white", fontweight="bold")
    bottom += heights
noise.set_xticks(np.arange(2))
noise.set_xticklabels(["catégorie\naccusation", "catégorie\ndécouverte"])
noise.set_ylabel("nombre de sujets")
noise.set_title(f"Le bruit d'étiquetage : {100 * agreement / len(entries):.0f} % d'accord seulement",
                fontsize=10)
noise.set_ylim(0, 292)
noise.legend(fontsize=7.5, loc="upper center", ncol=1)

# (b) Le taux de basculement, avant et après annotation.
labels = ["étiquette\nde catégorie", "registre\nannoté"]
values_a, values_d, pvalues = [], [], []
for key in ("category", "register"):
    group_a = [p for p in profiles if p[key] == "accusation"]
    group_d = [p for p in profiles if p[key] == "discovery"]
    ha, na, pa = switching_rate(group_a)
    hd, nd, pd = switching_rate(group_d)
    values_a.append(pa)
    values_d.append(pd)
    pvalues.append(stats.fisher_exact([[ha, na - ha], [hd, nd - hd]]).pvalue)
positions = np.arange(2)
rates.bar(positions - 0.18, values_a, 0.34, color=PALETTE["field"], label="accusation")
rates.bar(positions + 0.18, values_d, 0.34, color=PALETTE["remedy"], label="découverte")
for index, pvalue in enumerate(pvalues):
    height = max(values_a[index], values_d[index])
    rates.text(index, height + 0.5, f"p = {pvalue:.3f}", ha="center", fontsize=8,
               color=PALETTE["neutral"])
rates.set_xticks(positions)
rates.set_xticklabels(labels)
rates.set_ylabel("sujets ayant basculé  [%]")
rates.set_ylim(0, max(values_a + values_d) * 1.35)
rates.set_title("L'écart de taux disparaît avec l'étiquette", fontsize=10)
rates.legend(fontsize=8)

# (c) La persistance, par registre annoté.
groups = [np.array([s["lift"] for s in shifts if s["register"] == r]) for r in REGISTERS]
parts = persist.boxplot(groups, widths=0.5, patch_artist=True, showfliers=False)
for patch, register in zip(parts["boxes"], REGISTERS, strict=True):
    patch.set_facecolor(colours[register])
    patch.set_alpha(0.35)
for index, (values, register) in enumerate(zip(groups, REGISTERS, strict=True), start=1):
    jitter = np.linspace(-0.12, 0.12, values.size)
    persist.scatter(index + jitter, values, s=22, color=colours[register], zorder=3)
persist.set_xticks([1, 2, 3])
persist.set_xticklabels([f"{names[r]}\n(n={g.size})" for r, g in zip(REGISTERS, groups, strict=True)])
persist.set_ylabel("élévation durable du régime  [×]")
test = stats.mannwhitneyu(groups[0], groups[1], alternative="two-sided")
persist.set_title(f"La persistance ne diffère pas (p = {test.pvalue:.2f})", fontsize=10)

# (d) Le défaut de conception : composition en types de sujet.
shown = ["event", "concept", "person", "object", "organisation", "work"]
width = 0.38
offsets = {"accusation": -width / 2, "discovery": width / 2}
for register in ("accusation", "discovery"):
    heights = [kinds_by_register[register][k] for k in shown]
    design.barh(np.arange(len(shown)) + offsets[register], heights, width,
                color=colours[register], label=names[register])
design.set_yticks(np.arange(len(shown)))
design.set_yticklabels(["événement", "concept", "personne", "objet", "organisation", "œuvre"])
design.invert_yaxis()
design.set_xlabel("nombre de sujets")
design.set_title("Les deux registres ne portent pas sur les mêmes objets", fontsize=10)
design.legend(fontsize=8)

figure.suptitle("Annotation en aveugle : le bruit d'étiquetage portait l'écart", fontsize=12)
figure.tight_layout(rect=(0, 0, 1, 0.96))
save_figure(figure, "fig12_annotation")
plt.show()

## 7. Ce que le notebook établit

**Le bruit d'étiquetage était massif.** Deux sujets sur cinq ne relèvent d'aucun des deux
registres, et l'accord entre catégorie et annotation n'atteint que 59,5 %. La conjecture du
notebook 11 est vérifiée, et son ampleur dépasse ce qu'il supposait.

**Il portait la totalité de l'écart de taux de basculement.** Le rapport de cotes de 3,4
($p = 0{,}012$) devient 0,93 ($p = 1{,}00$) une fois l'étiquette corrigée. Les sujets écartés
basculent à 6,9 %, plus souvent que les deux registres : ce n'étaient pas des observations
inertes, c'étaient elles qui produisaient l'écart.

**La persistance reste nulle, et cette fois le test avait la puissance de conclure.** ×3,04
contre ×2,90, $p = 0{,}90$, robuste au retrait des sujets contaminés, des annotations
incertaines et des sujets à faible trafic. Un écart de l'ampleur annoncée par le corpus pilote
aurait été détecté.

> **La question laissée ouverte par le corpus étendu est tranchée : l'écart n'était pas dilué
> par l'étiquetage, il n'existe pas.**

**Et l'annotation expose un défaut de conception.** Correctement étiquetés, les deux registres
ne portent presque pas sur les mêmes types de sujets — concepts et événements d'un côté,
objets et personnes de l'autre. Une comparaison bâtie sur des catégories thématiques compare
donc aussi des natures d'objets. C'est une limite du plan d'expérience, pas du résultat : elle
ne ressuscite pas l'écart, elle indique ce qu'un quatrième protocole devrait contrôler.

**Ce que le dispositif ne couvre pas.** L'annotateur est unique : il n'y a pas d'accord
inter-juges, donc pas de mesure de la fiabilité du codage. La grille écrite et publiée est ce
qui rend le travail réplicable, non ce qui prouve qu'il serait reproduit à l'identique.

---

## 8. La réplication : deux codeurs indépendants

La section précédente laissait une réserve de méthode, et une seule : **l'annotateur était
unique**, donc rien ne mesurait la fiabilité du codage. Le corpus a donc été recodé par deux
lecteurs indépendants du contexte, sous la grille identique, à partir du même matériau —
présenté dans un ordre différent et **sans l'étiquette de catégorie**, pour qu'aucun codage ne
puisse recopier celui qu'il sert à vérifier.

In [11]:
from ide.annotation import cohen_kappa, consensus_registers, fleiss_kappa, load_replication

replication = load_replication()
codings = {"C1 — initial": annotations, **replication}

print(f"{'codeur':16s}" + "".join(f"{r:>13s}" for r in REGISTERS))
print("-" * 55)
for name, coding in codings.items():
    tally = Counter(item.register for item in coding.values())
    print(f"{name:16s}" + "".join(f"{tally[r]:13d}" for r in REGISTERS))

titles = sorted(annotations)
labels = {name: [coding[t].register for t in titles] for name, coding in codings.items()}

codeur             accusation    discovery      neither
-------------------------------------------------------
C1 — initial              147          118          175
C2-A                      140          114          186
C2-B                      141          109          190


Les distributions marginales sont déjà voisines. Reste à savoir si l'accord porte sur les
**mêmes sujets** — ce que seul un accord corrigé du hasard peut dire, sur un corpus dont 40 %
relèvent d'une seule étiquette.

In [12]:
names = list(codings)
print(f"{'paire':32s} {'accord brut':>12s} {'κ de Cohen':>12s}")
print("-" * 58)
for i, first in enumerate(names):
    for second in names[i + 1:]:
        left, right = labels[first], labels[second]
        raw = sum(1 for a, b in zip(left, right, strict=True) if a == b) / len(left)
        print(f"{first + '  vs  ' + second:32s} {100 * raw:11.1f} % {cohen_kappa(left, right):12.3f}")

three_way = fleiss_kappa(list(labels.values()))
print(f"\nκ de Fleiss, les trois codeurs ensemble : {three_way:.3f}")

unanimous = sum(1 for t in titles if len({coding[t].register for coding in codings.values()}) == 1)
print(f"unanimité sur {unanimous}/{len(titles)} = {100 * unanimous / len(titles):.1f} % des sujets")

paire                             accord brut   κ de Cohen
----------------------------------------------------------
C1 — initial  vs  C2-A                  93.6 %        0.903
C1 — initial  vs  C2-B                  94.5 %        0.917
C2-A  vs  C2-B                          96.4 %        0.944

κ de Fleiss, les trois codeurs ensemble : 0.921
unanimité sur 406/440 = 92.3 % des sujets


### Lecture, et la réserve qui compte

Sur l'échelle de Landis et Koch, $\kappa > 0{,}80$ se lit « accord presque parfait ». La grille
est donc **reproductible** : une lecture fraîche des mêmes consignes, sans accès au premier
codage ni aux résultats, redonne les mêmes étiquettes.

!!! danger "Ce que cet accord ne mesure pas"
    Les trois codeurs sont des instances du **même modèle de langue**. L'accord obtenu mesure
    la reproductibilité de la **grille**, pas l'accord entre juges humains indépendants — et il
    le surestime nécessairement, des instances d'un même modèle partageant leurs a priori. La
    réserve d'un codage humain multiple subsiste entière ; ce qui a changé, c'est qu'on sait
    désormais que les consignes écrites suffisent à produire un codage stable.

In [13]:
disagreements = [
    (t, tuple(sorted({coding[t].register for coding in codings.values()})))
    for t in titles
    if len({coding[t].register for coding in codings.values()}) > 1
]
patterns = Counter(pattern for _, pattern in disagreements)

print(f"{len(disagreements)} sujets non unanimes, par nature du désaccord :\n")
for pattern, count in patterns.most_common():
    swaps = "  ← inverse les deux registres comparés" if "neither" not in pattern else ""
    print(f"  {count:3d}  {' / '.join(pattern)}{swaps}")

crossing = sum(count for pattern, count in patterns.items() if "neither" not in pattern)
print(f"\nDésaccords portant sur l'appartenance à un registre : {len(disagreements) - crossing}")
print(f"Désaccords inversant accusation et découverte        : {crossing}")

34 sujets non unanimes, par nature du désaccord :

   17  discovery / neither
   16  accusation / neither
    1  accusation / discovery  ← inverse les deux registres comparés

Désaccords portant sur l'appartenance à un registre : 33
Désaccords inversant accusation et découverte        : 1


### Le désaccord est structuré, et il tombe au bon endroit

Presque tous les désaccords opposent un registre à « ni l'un ni l'autre » — c'est-à-dire qu'ils
portent sur **l'appartenance** d'un sujet à la comparaison, non sur le côté où le ranger. Un
seul sujet sur 440 voit un codeur dire « accusation » là où un autre dit « découverte ».

La conséquence est directe : l'ambiguïté résiduelle de la grille fait varier les **effectifs**
de la comparaison, pas son **sens**. C'est la forme d'imprécision la moins dommageable qu'on
pouvait espérer.

Les cas litigieux sont d'ailleurs interprétables : ce sont des étoiles dont le chapeau ne dit
pas si leur notabilité tient à une découverte, des dispositifs de physique sans annonce
associée, et quelques affaires dont le chapeau ne rapporte pas la mise en cause. La règle 4 de
la grille — l'objet de catalogue — est celle qui laisse le plus de latitude.

## 9. Le résultat tient-il sous le codage consensuel ?

C'est la seule question qui compte vraiment. Reprenons les deux tests avec, pour chaque sujet,
le registre **majoritaire des trois codeurs**.

In [14]:
consensus = consensus_registers(list(codings.values()))
print("registre consensuel :", dict(Counter(consensus.values())))

for record in profiles:
    record["consensus"] = consensus[record["subject"]]
for record in shifts:
    record["consensus"] = consensus[record["subject"]]

print(f"\n{'étiquette':24s} {'accusation':>16s} {'découverte':>16s} {'RC':>6s} {'p':>8s}")
print("-" * 74)
for key, label in (("category", "catégorie"),
                   ("register", "annotation C1"),
                   ("consensus", "consensus 3 codeurs")):
    group_a = [p for p in profiles if p[key] == "accusation"]
    group_d = [p for p in profiles if p[key] == "discovery"]
    ha, na, pa = switching_rate(group_a)
    hd, nd, pd = switching_rate(group_d)
    test = stats.fisher_exact([[ha, na - ha], [hd, nd - hd]])
    print(f"{label:24s} {f'{ha:3d}/{na:3d} = {pa:4.1f} %':>16s}"
          f" {f'{hd:3d}/{nd:3d} = {pd:4.1f} %':>16s}"
          f" {test.statistic:6.2f} {test.pvalue:8.3f}")

registre consensuel : {'neither': 188, 'accusation': 141, 'discovery': 111}

étiquette                      accusation       découverte     RC        p
--------------------------------------------------------------------------
catégorie                 19/220 =  8.6 %   6/220 =  2.7 %   3.37    0.012
annotation C1              7/147 =  4.8 %   6/118 =  5.1 %   0.93    1.000
consensus 3 codeurs        6/141 =  4.3 %   6/111 =  5.4 %   0.78    0.769


In [15]:
print(f"{'étiquette':24s} {'accusation':>14s} {'découverte':>14s} {'p':>8s}")
print("-" * 62)
for key, label in (("register", "annotation C1"), ("consensus", "consensus 3 codeurs")):
    lifts_a = np.array([s["lift"] for s in shifts if s[key] == "accusation"])
    lifts_d = np.array([s["lift"] for s in shifts if s[key] == "discovery"])
    test = stats.mannwhitneyu(lifts_a, lifts_d, alternative="two-sided")
    left = f"×{np.median(lifts_a):.2f} (n={lifts_a.size:2d})"
    right = f"×{np.median(lifts_d):.2f} (n={lifts_d.size:2d})"
    print(f"{label:24s} {left:>14s} {right:>14s} {test.pvalue:8.3f}")

print("\nLes deux résultats sont inchangés. Le codage initial n'était pas un cas particulier.")

étiquette                    accusation     découverte        p
--------------------------------------------------------------
annotation C1              ×3.04 (n= 7)   ×2.90 (n= 7)    0.902
consensus 3 codeurs        ×3.06 (n= 6)   ×2.90 (n= 7)    0.836

Les deux résultats sont inchangés. Le codage initial n'était pas un cas particulier.


In [16]:
figure, axes = plt.subplots(1, 3, figsize=(12.0, 3.9))
agreement_panel, structure, robustness = axes

# (a) Les accords deux à deux, et l'accord à trois.
pairs, values = [], []
for i, first in enumerate(names):
    for second in names[i + 1:]:
        pairs.append(f"{first.split(' —')[0].split(' ')[0]}\n{second}")
        values.append(cohen_kappa(labels[first], labels[second]))
pairs.append("trois codeurs\n(Fleiss)")
values.append(three_way)
colours = [PALETTE["remedy"]] * (len(values) - 1) + [PALETTE["order"]]
agreement_panel.bar(np.arange(len(values)), values, 0.6, color=colours, alpha=0.85)
agreement_panel.axhline(0.8, color=PALETTE["neutral"], linestyle="--", linewidth=1.2)
agreement_panel.text(-0.45, 1.10, "κ = 0,80 — seuil de l'accord presque parfait",
                     fontsize=7.5, color=PALETTE["neutral"])
for index, value in enumerate(values):
    agreement_panel.text(index, value + 0.02, f"{value:.3f}", ha="center", fontsize=8)
agreement_panel.set_xticks(np.arange(len(values)))
agreement_panel.set_xticklabels(pairs, fontsize=7)
agreement_panel.set_ylim(0, 1.20)
agreement_panel.set_ylabel("κ")
agreement_panel.set_title("La grille est reproductible", fontsize=10)

# (b) Où porte le désaccord.
kinds = ["appartenance\nà un registre", "inversion\naccusation / découverte"]
counts = [len(disagreements) - crossing, crossing]
structure.bar(kinds, counts, 0.5, color=[PALETTE["neutral"], PALETTE["disorder"]], alpha=0.85)
for index, count in enumerate(counts):
    structure.text(index, count + 0.6, str(count), ha="center", fontsize=9)
structure.set_ylabel("sujets non unanimes")
structure.set_ylim(0, max(counts) * 1.25)
structure.set_title(f"Le désaccord change l'effectif,\npas le sens ({len(titles)} sujets)",
                    fontsize=10)

# (c) Le résultat sous les trois étiquetages.
schemes = [("category", "catégorie"), ("register", "annotation C1"),
           ("consensus", "consensus")]
rates_a, rates_d, pvalues = [], [], []
for key, _ in schemes:
    ha, na, pa = switching_rate([p for p in profiles if p[key] == "accusation"])
    hd, nd, pd = switching_rate([p for p in profiles if p[key] == "discovery"])
    rates_a.append(pa)
    rates_d.append(pd)
    pvalues.append(stats.fisher_exact([[ha, na - ha], [hd, nd - hd]]).pvalue)
positions = np.arange(len(schemes))
robustness.bar(positions - 0.18, rates_a, 0.34, color=PALETTE["field"], label="accusation")
robustness.bar(positions + 0.18, rates_d, 0.34, color=PALETTE["remedy"], label="découverte")
for index, pvalue in enumerate(pvalues):
    robustness.text(index, max(rates_a[index], rates_d[index]) + 0.35,
                    f"p = {pvalue:.3f}", ha="center", fontsize=7.5, color=PALETTE["neutral"])
robustness.set_xticks(positions)
robustness.set_xticklabels([label for _, label in schemes], fontsize=8)
robustness.set_ylabel("sujets ayant basculé  [%]")
robustness.set_ylim(0, max(rates_a + rates_d) * 1.35)
robustness.set_title("Le résultat ne dépend pas du codeur", fontsize=10)
robustness.legend(fontsize=8)

figure.suptitle("Réplication : κ de Fleiss = %.3f, et un résultat inchangé" % three_way,
                fontsize=12)
figure.tight_layout(rect=(0, 0, 1, 0.93))
save_figure(figure, "fig12b_replication")
plt.show()

## 10. Ce que la réplication ajoute

**La grille est reproductible** — $\kappa$ de Fleiss de 0,92, unanimité sur 92 % des sujets.
Les consignes écrites suffisent à produire un codage stable, ce qui rend le travail
réplicable par un tiers plutôt que seulement consultable.

**Le désaccord résiduel tombe au bon endroit.** Un seul sujet sur 440 voit deux codeurs
inverser les registres comparés ; tout le reste porte sur l'appartenance à la comparaison.
L'imprécision fait varier des effectifs, pas un sens.

**Et le résultat est inchangé** sous le codage consensuel : le taux de basculement reste sans
écart, la persistance aussi. Le codage initial n'était pas un cas particulier.

> **La dernière réserve de méthode identifiable sans juges humains est levée.** Ce qui subsiste
> — que trois instances d'un même modèle ne valent pas trois juges indépendants — ne se lèvera
> qu'avec des annotateurs humains, et ce n'est plus une question de calcul.

## Pistes ouvertes

1. **Faire annoter le même corpus par un second codeur**, à l'aveugle également, et publier le
   $\kappa$ de Cohen. C'est le complément direct et peu coûteux de ce travail.
2. **Apparier sur le type de sujet** autant que sur le trafic, dès la construction du corpus.
   Comparer des événements à des événements et des personnes à des personnes est désormais
   une exigence mesurée, non une précaution théorique.
3. **Abaisser le seuil de détection en agrégeant par semaine.** Vingt-huit basculements sur
   440 sujets laissent le test principal à sept observations par registre ; c'est le facteur
   limitant qui reste.
4. **Chercher l'effet ailleurs que dans la persistance.** Trois quantités ont été testées —
   taux d'amplification, persistance, taux de basculement — et aucune ne distingue les
   registres. Si le mécanisme de la charge émotionnelle existe, il ne se lit pas dans la
   dynamique d'attention agrégée d'une encyclopédie.